# Проект 1. Влияние повышения цен на ПРАЙМ в mobile-сегменте

### Бизнес-вопрос: Было ли повышение цен на ПРАЙМ в марте успешным по выручке с подписок mobile-сегмента с учётом изменения платёжного объёма?

### Бизнес-гипотеза: Повышение цен увеличит выручку mobile-сегмента, несмотря на снижения количества операций.

Для проверки сравним два равных окна до и после повышения цен, разложим изменение выручки по модели `R = N × P` на эффект количества платежей и эффект цены и оценим, насколько фактическое снижение платежей близко к границе, при которой новая цена перестаёт компенсировать потерю объёма.

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv

# Находим .env в корне проекта
env_path = find_dotenv(usecwd=True)

if not env_path:
    raise SystemExit(
        "Файл .env не найден.\n"
        "Создайте его в корне репозитория рядом с README.md."
    )

# Загружаем переменные из .env
load_dotenv(env_path)

# Берём строку подключения
dsn = os.environ.get("PRIME_DSN")

if not dsn:
    raise SystemExit(
        f"Файл {env_path} найден, но переменной PRIME_DSN в нём нет."
    )

print("Файл .env найден")

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(dsn)

pd.read_sql(
    "select current_user, current_database(), version()",
    engine
).T

#### Определение сегмента и периода исследования

Mobile-сегмент — клиенты, у которых была хотя бы одна сессия с `device_type = 'mobile'`
в фиксированном периоде времени до повышения цен.

Состав сегмента фиксируется до изменения цены и затем не меняется при сравнении
периодов до и после.

Для сравнения используются два равных 28-дневных окна:
- до повышения цен: 01.02.2026–28.02.2026;
- после повышения цен: 01.03.2026–28.03.2026.

In [ ]:
# Период, по которому определяем mobile-сегмент

SEGMENT_START = "2026-01-01"
SEGMENT_END = "2026-03-01"

# Равные периоды для сравнения до и после повышения цены

BEFORE_START = "2026-02-01"
BEFORE_END = "2026-03-01"

AFTER_START = "2026-03-01"
AFTER_END = "2026-03-29"

DEVICE_TYPE = "mobile"

#### Формирование mobile-сегмента и характеристики клиентов

In [ ]:
# Находим клиентов, которые пользовались mobile
# до повышения цен

mobile_clients = pd.read_sql(
    text("""
        select distinct client_id
        from prime.service_usage
        where device_type = :device_type
          and started_at >= :segment_start
          and started_at < :segment_end
        order by client_id
    """),
    engine,
    params={
        "device_type": DEVICE_TYPE,
        "segment_start": SEGMENT_START,
        "segment_end": SEGMENT_END,
    }
)

print("ЗЕРНО: одна строка — один клиент.")
print("Клиентов в mobile-сегменте:", len(mobile_clients))
print("client_id уникальны:", mobile_clients["client_id"].is_unique)

mobile_clients.head()

In [ ]:
# Загружаем характеристики клиентов mobile-сегмента

clients = pd.read_sql(
    text("""
        select
            c.client_id,
            c.acquisition_channel,
            c.region_code
        from prime.clients c
        where c.client_id in (
            select distinct client_id
            from prime.service_usage
            where device_type = :device_type
              and started_at >= :segment_start
              and started_at < :segment_end
        )
        order by c.client_id
    """),
    engine,
    params={
        "device_type": DEVICE_TYPE,
        "segment_start": SEGMENT_START,
        "segment_end": SEGMENT_END,
    }
)

print("Строк:", len(clients))
print("Совпадает с mobile_clients:", len(clients) == len(mobile_clients))

clients.head()

### Подготовка данных о подписках и платежах

#### Подписки mobile-клиентов


In [ ]:
# Загружаем подписки, которыми владеют клиенты mobile-сегмента

subs = pd.read_sql(
    text("""
        select
            s.subscription_id,
            s.owner_client_id,
            s.tariff_id
        from prime.subscriptions s
        where s.owner_client_id in (
            select distinct client_id
            from prime.service_usage
            where device_type = :device_type
              and started_at >= :segment_start
              and started_at < :segment_end
        )
        order by s.subscription_id
    """),
    engine,
    params={
        "device_type": DEVICE_TYPE,
        "segment_start": SEGMENT_START,
        "segment_end": SEGMENT_END,
    }
)

# Проверяем зерно таблицы подписок
print("Подписок:", len(subs))
print("subscription_id уникальны:", subs["subscription_id"].is_unique)
print("Владельцев с подпиской:", subs["owner_client_id"].nunique())
print("owner_client_id уникальны:", subs["owner_client_id"].is_unique)

subs.head()

В mobile-сегменте 193 943 клиента, из них 153 595 владеют собственной подпиской. В выгрузке каждому владельцу соответствует ровно одна подписка: `subscription_id` и `owner_client_id` уникальны.

Таким образом, 40 348 клиентов mobile-сегмента не имеют собственной подписки. Для анализа подписочной выручки рассматриваем платежи по подпискам, владельцами которых являются клиенты mobile-сегмента. Платёж не размножаем на других участников shared-подписки, чтобы не получить двойной счёт.

#### Успешные платежи до и после повышения цен

In [ ]:
# Загружаем успешные платежи по подпискам mobile-клиентов
# только за период исследования

pay = pd.read_sql(
    text("""
        select
            payment_id,
            subscription_id,
            paid_at,
            amount
        from prime.payments
        where status = 'success'
          and paid_at >= :before_start
          and paid_at < :after_end
          and subscription_id in (
              select subscription_id
              from prime.subscriptions
              where owner_client_id in (
                  select distinct client_id
                  from prime.service_usage
                  where device_type = :device_type
                    and started_at >= :segment_start
                    and started_at < :segment_end
              )
          )
        order by paid_at, payment_id
    """),
    engine,
    params={
        "before_start": BEFORE_START,
        "after_end": AFTER_END,
        "device_type": DEVICE_TYPE,
        "segment_start": SEGMENT_START,
        "segment_end": SEGMENT_END,
    }
)

# Выделяем календарный день платежа
pay["day"] = pay["paid_at"].dt.date

# Делим платежи на периоды до и после повышения цены
pay["period"] = "before"
pay.loc[
    pay["day"] >= pd.Timestamp(AFTER_START).date(),
    "period"
] = "after"

# Проверяем зерно: одна строка — один платеж
print("Платежей:", len(pay))
print("payment_id уникальны:", pay["payment_id"].is_unique)

# Проверяем, что в каждом окне есть все 28 дней
payment_check = (
    pay.groupby("period", as_index=False)
    .agg(
        n_payments=("payment_id", "size"),
        revenue=("amount", "sum"),
        n_days=("day", "nunique"),
    )
)

payment_check = payment_check.set_index("period").loc[["before", "after"]].reset_index()

payment_check

In [ ]:
# Берём выручку и число дней до повышения цены
before_revenue = payment_check.loc[
    payment_check["period"] == "before",
    "revenue"
].iloc[0]

before_days = payment_check.loc[
    payment_check["period"] == "before",
    "n_days"
].iloc[0]

# Берём выручку и число дней после повышения цены
after_revenue = payment_check.loc[
    payment_check["period"] == "after",
    "revenue"
].iloc[0]

after_days = payment_check.loc[
    payment_check["period"] == "after",
    "n_days"
].iloc[0]


# Считаем среднюю дневную выручку
before_mean = before_revenue / before_days
after_mean = after_revenue / after_days


# Считаем изменение средней дневной выручки
revenue_growth_pct = (
    after_mean / before_mean - 1
) * 100


print("Средняя дневная выручка до:", round(before_mean, 2))
print("Средняя дневная выручка после:", round(after_mean, 2))
print("Изменение:", round(revenue_growth_pct, 1), "%")

Оба периода содержат по 28 дней. После повышения цен количество успешных платежей немного снизилось: с 102 267 до 101 436, при этом выручка выросла с 28,13 млн ₽ до 32,98 млн ₽, а средняя дневная выручка — примерно с 1,00 млн до 1,18 млн руб.

Рост выручки произошёл несмотря на небольшое снижение платёжного объёма. Средняя дневная выручка после повышения цен выросла на 17,2%.

#### Метрики до и после изменения цен по тарифам

In [ ]:
# Присоединяем к каждому платежу тариф подписки

pay_tariff = pay.merge(
    subs[["subscription_id", "tariff_id"]],
    on="subscription_id",
    how="left",
    validate="m:1"
)

# Проверяем, что merge не размножил платежи и деньги

print("Строк до merge:", len(pay))
print("Строк после merge:", len(pay_tariff))
print("Выручка до merge:", pay["amount"].sum())
print("Выручка после merge:", pay_tariff["amount"].sum())
print("Тариф не найден:", pay_tariff["tariff_id"].isna().sum())

In [ ]:
# Считаем основные метрики по тарифу и периоду

tariff_metrics = (
    pay_tariff
    .groupby(["tariff_id", "period"], as_index=False)
    .agg(
        revenue=("amount", "sum"),
        n_payments=("payment_id", "size")
    )
)

# Средний успешный платеж

tariff_metrics["avg_payment"] = (
    tariff_metrics["revenue"] / tariff_metrics["n_payments"]
)

# Для удобства сортируем: сначала before, потом after

period_order = pd.CategoricalDtype(
    categories=["before", "after"],
    ordered=True
)

tariff_metrics["period"] = tariff_metrics["period"].astype(period_order)

tariff_metrics = tariff_metrics.sort_values(
    ["tariff_id", "period"]
).reset_index(drop=True)

tariff_metrics

После повышения цен число успешных платежей немного снизилось на обоих тарифах, однако выручка выросла. У «Базового» тарифа количество платежей сократилось с 63 357 до 62 811, а средний успешный платёж вырос с 199 до 249 ₽. У «Расширенного» количество платежей снизилось с 38 910 до 38 625, а средний платёж вырос с 399 до 449 ₽.

Таким образом, предварительно рост выручки связан с увеличением суммы одного платежа, которое оказалось весомее небольшого снижения платёжного объёма.

#### Проверка изменения цен

In [ ]:
# Загружаем историю цен тарифов

tariffs = pd.read_sql(
    text("""
        select
            tariff_id,
            monthly_fee,
            valid_from,
            valid_to
        from prime.tariffs
        order by tariff_id, valid_from
    """),
    engine
)

tariffs

В справочнике тарифов обнаружена проблемная строка: для тарифа 2 цена 429 ₽ действует с 01.02.2026 по 30.06.2026 и пересекается с интервалами цен 399 ₽ и 449 ₽.

Такое пересечение означает, что одному платежу потенциально могут соответствовать сразу несколько цен.

In [ ]:
# Убираем ошибочную строку с ценой 429 ₽ (семинар 4)

tariffs_fixed = tariffs[tariffs["monthly_fee"] != 429].copy()

# Даты тарифов приводим к UTC,
# чтобы их можно было сравнивать с paid_at

# Приводим даты к единому формату UTC
for col in ["valid_from", "valid_to"]:
    tariffs_fixed[col] = pd.to_datetime(
        tariffs_fixed[col],
        utc=True
    )

tariffs_fixed

In [ ]:
# Соединяем платежи со всеми строками их тарифа

priced = pay_tariff.merge(
    tariffs_fixed[
        ["tariff_id", "monthly_fee", "valid_from", "valid_to"]
    ],
    on="tariff_id",
    how="inner",
    validate="m:m"
)

# Оставляем только ту цену, которая действовала
# в момент конкретного платежа

priced = priced[
    (priced["paid_at"] >= priced["valid_from"])
    & (
        priced["paid_at"]
        < priced["valid_to"] + pd.Timedelta(days=1)
    )
]

print("Платежей до:", len(pay_tariff))
print("Платежей с найденной ценой:", len(priced))
print("Выручка до:", pay_tariff["amount"].sum())
print("Выручка после:", priced["amount"].sum())
print(
    "Платежей, где amount не совпал с ценой:",
    (priced["amount"] != priced["monthly_fee"]).sum()
)

После исключения ошибочной строки с ценой 429 ₽ каждому из 203 703 платежей соответствует ровно одна действующая цена тарифа. Число платежей и общая выручка после присоединения справочника не изменились, а сумма каждого платежа совпадает с найденной тарифной ценой.

Дальнейшее изменение среднего платежа можно интерпретировать именно как эффект изменения цены тарифа, а не как результат ошибки соединения или структуры данных.

#### Факторная декомпозиция

In [ ]:
# Разделяем изменение выручки на два фактора:
# 1. изменение количества платежей
# 2. изменение среднего платежа

def chain(n0, p0, n1, p1):
    # Эффект изменения количества платежей
    volume_effect = (n1 - n0) * p0

    # Эффект изменения среднего платежа
    price_effect = n1 * (p1 - p0)

    # Проверяем, что два эффекта полностью объясняют изменение выручки
    revenue_change = n1 * p1 - n0 * p0

    assert abs(
        volume_effect + price_effect - revenue_change
    ) < 0.01

    return volume_effect, price_effect

In [ ]:
# Собираем результаты факторного анализа

decomposition = []

for tariff_id in tariff_metrics["tariff_id"].unique():
    t = tariff_metrics[tariff_metrics["tariff_id"] == tariff_id]

    before = t[t["period"] == "before"].iloc[0]
    after = t[t["period"] == "after"].iloc[0]

    volume_effect, price_effect = chain(
        before["n_payments"],
        before["avg_payment"],
        after["n_payments"],
        after["avg_payment"]
    )

    decomposition.append({
        "tariff_id": tariff_id,
        "revenue_before": before["revenue"],
        "revenue_after": after["revenue"],
        "revenue_change": after["revenue"] - before["revenue"],
        "volume_effect": volume_effect,
        "price_effect": price_effect
    })

decomposition = pd.DataFrame(decomposition)

decomposition

Факторная декомпозиция показывает, что рост выручки по обоим тарифам обеспечен повышением цены, тогда как изменение платёжного объёма дало небольшой отрицательный эффект.

Для «Базового» тарифа снижение количества платежей уменьшило выручку на 108 654 ₽, но повышение цены добавило 3 140 550 ₽, поэтому итоговая выручка выросла на 3 031 896 ₽. Для «Расширенного» отрицательный эффект объёма составил 113 715 ₽, а положительный эффект цены — 1 931 250 ₽, что дало итоговый рост на 1 817 535 ₽.

### Граница снижения платёжного объёма

Факторная модель курса задаёт выручку как `R = N × P`, где `N` — число платежей, а `P` — средний платёж.

Чтобы найти границу снижения объёма, при которой новая цена ещё сохраняет прежнюю выручку, приравниваем выручку до и после:

`N₁ × P₁ = N₀ × P₀`.

Отсюда `N₁ / N₀ = P₀ / P₁`, поэтому максимальная доля снижения числа платежей равна: `1 - P₀ / P₁`.

Сравним эту границу с фактическим снижением платежей.

In [ ]:
# Считаем фактическое и предельно допустимое
# снижение количества платежей по каждому тарифу

volume_limits = []

for tariff_id in tariff_metrics["tariff_id"].unique():
    t = tariff_metrics[tariff_metrics["tariff_id"] == tariff_id]

    before = t[t["period"] == "before"].iloc[0]
    after = t[t["period"] == "after"].iloc[0]

    # Фактическое снижение количества платежей
    actual_decline = (
        1 - after["n_payments"] / before["n_payments"]
    )

    # Максимальное снижение объёма,
    # при котором новая цена ещё сохраняет прежнюю выручку
    allowed_decline = (
        1 - before["avg_payment"] / after["avg_payment"]
    )

    # Запас до этой границы, в процентных пунктах
    reserve_pp = (
        allowed_decline - actual_decline
    ) * 100

    volume_limits.append({
        "tariff_id": tariff_id,
        "actual_decline_pct": actual_decline * 100,
        "allowed_decline_pct": allowed_decline * 100,
        "reserve_pp": reserve_pp
    })

volume_limits = pd.DataFrame(volume_limits)

volume_limits.round(2)

Фактическое снижение платёжного объёма оказалось значительно меньше рассчитанной границы. Для «Базового» тарифа платежи снизились на 0,86% при границе около 20,08%, для «Расширенного» — на 0,73% при границе около 11,14%.

То есть в исследуемых 28-дневных окнах оба тарифа остаются далеко от уровня снижения платежей, при котором положительный эффект новой цены на выручку был бы полностью потерян.

#### Диагностика платёжного объёма по каналам привлечения

In [ ]:
# Добавляем владельца подписки к каждому платежу

pay_channel = pay_tariff.merge(
    subs[["subscription_id", "owner_client_id"]],
    on="subscription_id",
    how="left",
    validate="m:1"
)

# Добавляем канал привлечения владельца

pay_channel = pay_channel.merge(
    clients[["client_id", "acquisition_channel"]],
    left_on="owner_client_id",
    right_on="client_id",
    how="left",
    validate="m:1"
)

# Проверяем, что соединения не размножили платежи и деньги

print("Строк до:", len(pay_tariff))
print("Строк после:", len(pay_channel))

print("Выручка до:", pay_tariff["amount"].sum())
print("Выручка после:", pay_channel["amount"].sum())

print(
    "Платежей без известного канала:",
    pay_channel["acquisition_channel"].isna().sum()
)

In [ ]:
# Считаем платежи отдельно по тарифу, каналу и периоду

channel_payments = (
    pay_channel
    .groupby(
        ["tariff_id", "acquisition_channel", "period"],
        as_index=False,
        dropna=False
    )
    .agg(
        n_payments=("payment_id", "size")
    )
)

# Для удобства явно называем пропуски

channel_payments["acquisition_channel"] = (
    channel_payments["acquisition_channel"]
    .fillna("без канала")
)

# Задаём порядок периодов: сначала before, потом after

channel_payments["period"] = pd.Categorical(
    channel_payments["period"],
    categories=["before", "after"],
    ordered=True
)

# Сортируем таблицу
channel_payments = channel_payments.sort_values(
    ["tariff_id", "acquisition_channel", "period"]
).reset_index(drop=True)

channel_payments.head(30)

In [ ]:
# Ставим before и after рядом для каждого тарифа и канала

channel_change_base = (
    channel_payments
    .pivot(
        index=["tariff_id", "acquisition_channel"],
        columns="period",
        values="n_payments"
    )
    .reset_index()
)

channel_change_base.columns.name = None

# Явно задаём удобный порядок колонок
channel_change_base = channel_change_base[
    ["tariff_id", "acquisition_channel", "before", "after"]
]

# Считаем фактическое снижение платежей
# Положительное число = платежей стало меньше

channel_change_base["actual_decline_pct"] = (
    1
    - channel_change_base["after"] / channel_change_base["before"]
) * 100

channel_change_base

In [ ]:
# Добавляем предельно допустимое снижение
# для соответствующего тарифа

channel_change = channel_change_base.merge(
    volume_limits[
        ["tariff_id", "allowed_decline_pct"]
    ],
    on="tariff_id",
    how="left",
    validate="m:1"
)

# Считаем запас до границы
channel_change["reserve_pp"] = (
    channel_change["allowed_decline_pct"]
    - channel_change["actual_decline_pct"]
)

# Сначала показываем наиболее рискованные каналы

channel_change = (
    channel_change
    .sort_values(
        ["tariff_id", "reserve_pp"]
    )
    .reset_index(drop=True)
)

# Финальный порядок колонок
channel_change = channel_change[
    [
        "tariff_id",
        "acquisition_channel",
        "before",
        "after",
        "actual_decline_pct",
        "allowed_decline_pct",
        "reserve_pp"
    ]
]

channel_change.round(2)

Во всех каналах фактическое снижение платёжного объёма остаётся ниже предельно допустимого уровня для соответствующего тарифа. Наиболее заметное снижение наблюдается в канале `social`: на 2,68% для тарифа 1 и на 3,21% для тарифа 2.

При этом запас до предельно допустимой границы составляет 17,40 п.п. для тарифа 1 и 7,93 п.п. для тарифа 2. Даже в наиболее чувствительном канале снижение количества платежей не перекрывает положительный эффект повышения цены.

В каналах `crm` и `referral` для тарифа 2 количество платежей, наоборот, немного выросло, а в `organic` и среди клиентов без известного канала практически не изменилось. Выраженного проблемного канала, который ставил бы общий результат повышения цен под сомнение, в рассматриваемом периоде не обнаружено; `social` имеет смысл выделить для дальнейшего мониторинга.

### Визуализация результатов

#### График 1. Что произошло с выручкой после 1 марта

In [ ]:
# Считаем дневную выручку
daily_revenue = (
    pay
    .groupby("day", as_index=False)
    .agg(
        revenue=("amount", "sum")
    )
)

daily_revenue["day"] = pd.to_datetime(daily_revenue["day"])

daily_revenue.head()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter

# Спокойная тема
sns.set_theme(style="whitegrid", context="notebook")

# Считаем дневную выручку
daily_revenue = (
    pay
    .groupby("day", as_index=False)
    .agg(
        revenue=("amount", "sum")
    )
)

daily_revenue["day"] = pd.to_datetime(daily_revenue["day"])

# Средняя дневная выручка до повышения цены
before_mean = daily_revenue.loc[
    daily_revenue["day"] < pd.Timestamp(AFTER_START),
    "revenue"
].mean()

# Средняя дневная выручка после повышения цены
after_mean = daily_revenue.loc[
    daily_revenue["day"] >= pd.Timestamp(AFTER_START),
    "revenue"
].mean()

# Рост средней дневной выручки
revenue_growth_pct = (
    after_mean / before_mean - 1
) * 100


fig, ax = plt.subplots(figsize=(12, 5))

# Основной дневной ряд
sns.lineplot(
    data=daily_revenue,
    x="day",
    y="revenue",
    ax=ax,
    linewidth=2.2,
    color="#3568A8"
)

# Дата повышения цены
ax.axvline(
    pd.Timestamp(AFTER_START),
    color="#D97706",
    linestyle="--",
    linewidth=1.5,
    alpha=0.9
)

# Средняя выручка ДО
ax.hlines(
    before_mean,
    xmin=pd.Timestamp(BEFORE_START),
    xmax=pd.Timestamp(BEFORE_END),
    color="#6B7280",
    linestyles=":",
    linewidth=2
)

# Средняя выручка ПОСЛЕ
ax.hlines(
    after_mean,
    xmin=pd.Timestamp(AFTER_START),
    xmax=pd.Timestamp(AFTER_END),
    color="#6B7280",
    linestyles=":",
    linewidth=2
)

# Подпись повышения цены
ax.annotate(
    "Повышение цен",
    xy=(pd.Timestamp(AFTER_START), after_mean),
    xytext=(10, 30),
    textcoords="offset points",
    color="#D97706",
    fontsize=10
)

# Подпись среднего ДО — выше пунктирной линии
ax.annotate(
    f"среднее: {before_mean / 1_000_000:.2f} млн",
    xy=(pd.Timestamp("2026-02-03"), before_mean),
    xytext=(0, 25),
    textcoords="offset points",
    color="#6B7280",
    fontsize=9
)

# Подпись среднего ПОСЛЕ — выше пунктирной линии
ax.annotate(
    f"среднее: {after_mean / 1_000_000:.2f} млн",
    xy=(pd.Timestamp("2026-03-03"), after_mean),
    xytext=(-15, 15),
    textcoords="offset points",
    color="#6B7280",
    fontsize=9
)

# Ось Y в миллионах
ax.yaxis.set_major_formatter(
    FuncFormatter(
        lambda x, _: f"{x / 1_000_000:.2f}"
    )
)

# Даты в полном формате
ax.xaxis.set_major_locator(
    mdates.WeekdayLocator(interval=1)
)

ax.xaxis.set_major_formatter(
    mdates.DateFormatter("%Y-%m-%d")
)

# Горизонтальная сетка
ax.grid(
    axis="y",
    alpha=0.28,
    linewidth=0.9
)

# Вертикальная сетка чуть слабее
ax.grid(
    axis="x",
    alpha=0.18,
    linewidth=0.8
)

# Заголовок сразу сообщает результат
ax.set_title(
    f"Средняя дневная выручка выросла на {revenue_growth_pct:.1f}% после повышения цен",
    loc="left",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Дата")
ax.set_ylabel("Выручка, млн руб.")

sns.despine()

plt.tight_layout()
plt.show()

После повышения цен средняя дневная выручка mobile-сегмента выросла на 17,2% — примерно с 1,00 млн до 1,18 млн руб. в день.

Рост виден не только в суммарных значениях двух 28-дневных окон, но и в дневной динамике: после 1 марта ряд в целом сместился на более высокий уровень. При этом внутри каждого периода сохраняются ежедневные колебания.

#### График 2. Платёжный объём до и после изменения цен по тарифам

In [ ]:
# Копия таблицы для графика
payment_plot = tariff_metrics.copy()

# Переименовываем тарифы по-русски
payment_plot["tariff"] = payment_plot["tariff_id"].map({
    1: "Базовый",
    2: "Расширенный"
})

# Делаем понятные подписи периода для легенды
payment_plot["period_label"] = payment_plot["period"].map({
    "before": "До повышения",
    "after": "После повышения"
})

# Порядок отображения
period_order = ["До повышения", "После повышения"]
tariff_order = ["Базовый", "Расширенный"]

plt.figure(figsize=(9, 5))
ax = sns.barplot(
    data=payment_plot,
    x="tariff",
    y="n_payments",
    hue="period_label",
    order=tariff_order,
    hue_order=period_order,
    palette=["#4C78A8", "#F28E2B"],
    edgecolor="white",
    linewidth=1.2
)

# Подписи значений на столбиках
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", padding=3, fontsize=9)

ax.set_title(
    "После повышения цен платёжный объём почти не изменился",
    loc="left",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel("Тариф")
ax.set_ylabel("Количество успешных платежей")
ax.set_ylim(bottom=0)

# Делаем легенду более понятной и заметной
ax.legend(
    title="Период сравнения",
    loc="upper right",
    frameon=True,
    facecolor="white",
    edgecolor="lightgray"
)

# Сетка
ax.grid(axis="y", alpha=0.3, linewidth=0.9)
ax.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

После повышения цен количество успешных платежей снизилось на обоих тарифах. Для «Базового» число платежей уменьшилось с 63 357 до 62 811 — примерно на 0,86%, для «Расширенного» — с 38 910 до 38 625, примерно на 0,73%.

Таким образом, рост выручки после повышения цен не сопровождался резким сокращением платёжного объёма.

#### Факторная декомпозиция изменения выручки

In [ ]:
# Копируем таблицу для графика
plot_df = decomposition.copy()

# Названия тарифов
tariff_names = {
    1: "Базовый",
    2: "Расширенный"
}

plot_df["tariff_name"] = plot_df["tariff_id"].map(tariff_names)

# Переводим значения в млн руб.
for col in [
    "revenue_before",
    "revenue_after",
    "volume_effect",
    "price_effect"
]:
    plot_df[col + "_mln"] = plot_df[col] / 1_000_000


# Цвета
base_color = "#4C78A8"
neg_color = "#E45756"
pos_color = "#F28E2B"
line_color = "#9CA3AF"

sns.set_theme(style="whitegrid", context="notebook")


# Одинаковая шкала для обоих тарифов
max_revenue = plot_df[
    ["revenue_before_mln", "revenue_after_mln"]
].max().max()

y_max = max_revenue * 1.12


fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5.5),
    sharey=True
)


def draw_waterfall(ax, row):

    before = row["revenue_before_mln"]
    volume = row["volume_effect_mln"]
    price = row["price_effect_mln"]
    after = row["revenue_after_mln"]

    after_volume = before + volume
    after_price = after_volume + price

    x = [0, 1, 2, 3]
    width = 0.72

    labels = [
        "До повышения",
        "Эффект\nобъёма",
        "Эффект\nцены",
        "После повышения"
    ]

    # Выручка до
    ax.bar(
        x[0],
        before,
        width=width,
        color=base_color
    )

    # Эффект объёма
    volume_bottom = min(before, after_volume)

    ax.bar(
        x[1],
        abs(volume),
        bottom=volume_bottom,
        width=width,
        color=neg_color
    )

    # Эффект цены
    price_bottom = min(after_volume, after_price)

    ax.bar(
        x[2],
        abs(price),
        bottom=price_bottom,
        width=width,
        color=pos_color
    )

    # Выручка после
    ax.bar(
        x[3],
        after,
        width=width,
        color=base_color
    )


    # Соединительные линии
    ax.hlines(
        before,
        x[0] + width / 2,
        x[1] - width / 2,
        color=line_color,
        linestyle="--",
        linewidth=1.3
    )

    ax.hlines(
        after_volume,
        x[1] + width / 2,
        x[2] - width / 2,
        color=line_color,
        linestyle="--",
        linewidth=1.3
    )

    ax.hlines(
        after,
        x[2] + width / 2,
        x[3] - width / 2,
        color=line_color,
        linestyle="--",
        linewidth=1.3
    )


    # Подписи начального и итогового значения
    ax.text(
        x[0],
        before + 0.25,
        f"{before:.2f}",
        ha="center",
        fontweight="bold",
        color=base_color
    )

    ax.text(
        x[3],
        after + 0.25,
        f"{after:.2f}",
        ha="center",
        fontweight="bold",
        color=base_color
    )


    # Эффект объёма:
    # значение и стрелка делают маленький минус заметным,
    # но сам столбик остаётся пропорционален данным
    ax.annotate(
        f"{volume:+.2f}",
        xy=(
            x[1],
            after_volume
        ),
        xytext=(
            x[1],
            after_volume + 1.0
        ),
        ha="center",
        color=neg_color,
        fontweight="bold",
        arrowprops={
            "arrowstyle": "-|>",
            "color": neg_color,
            "lw": 1.4
        }
    )


    # Эффект цены
    ax.annotate(
        f"{price:+.2f}",
        xy=(
            x[2],
            after_price
        ),
        xytext=(
            x[2],
            after_price + 0.7
        ),
        ha="center",
        color=pos_color,
        fontweight="bold",
        arrowprops={
            "arrowstyle": "-|>",
            "color": pos_color,
            "lw": 1.4
        }
    )


    ax.set_xticks(x)
    ax.set_xticklabels(labels)

    ax.set_title(
        row["tariff_name"],
        fontsize=13,
        fontweight="bold"
    )

    # Одинаковая шкала для обоих графиков
    ax.set_ylim(0, y_max)

    ax.grid(
        axis="y",
        alpha=0.22,
        linewidth=0.8
    )

    ax.grid(
        axis="x",
        visible=False
    )

    sns.despine(ax=ax)


# Строим оба тарифа
for ax, (_, row) in zip(
    axes,
    plot_df.iterrows()
):
    draw_waterfall(ax, row)


# Подписи осей
axes[0].set_ylabel("Выручка, млн руб.")
axes[1].set_ylabel("")


fig.suptitle(
    "Повышение цены перекрыло небольшой отрицательный эффект снижения платежей",
    fontsize=16,
    fontweight="bold",
    x=0.06,
    ha="left"
)

plt.tight_layout()
plt.show()

Факторная декомпозиция показывает, что рост выручки по обоим тарифам был обеспечен повышением цены, снижение количества платежей дало небольшой отрицательный эффект.

Для «Базового» тарифа снижение платёжного объёма уменьшило выручку примерно на 0,11 млн руб., но эффект повышения цены составил +3,14 млн руб. Для «Расширенного» отрицательный эффект объёма также составил около 0,11 млн руб., а положительный ценовой эффект — +1,93 млн руб.

Таким образом, итоговая выручка выросла по обоим тарифам.

#### Фактическое и предельно допустимое снижение платёжного объёма

In [ ]:
# Готовим данные
plot_df = volume_limits.copy()

# Названия тарифов
plot_df["tariff_name"] = plot_df["tariff_id"].map({
    1: "Базовый",
    2: "Расширенный"
})

plot_df = (
    plot_df
    .sort_values("tariff_id")
    .reset_index(drop=True)
)

# Цвета
actual_color = "#4C78A8"      # фактическое снижение
limit_color = "#F28E2B"       # граница
positive_color = "#DDEEDB"    # зона, где остаёмся в плюсе
negative_color = "#F8D7DA"    # зона после границы
positive_text = "#4F6350"
negative_text = "#A33A3A"

sns.set_theme(style="whitegrid", context="notebook")

fig, ax = plt.subplots(figsize=(12, 5.8))

# Базовый сверху, Расширенный снизу
y_positions = {
    "Базовый": 1,
    "Расширенный": 0
}

# Общая правая граница графика
x_max = plot_df["allowed_decline_pct"].max() + 2.5


for _, row in plot_df.iterrows():

    tariff = row["tariff_name"]
    y = y_positions[tariff]

    actual = row["actual_decline_pct"]
    allowed = row["allowed_decline_pct"]

    # Зелёная зона:
    # повышение цены ещё компенсирует снижение платежей
    ax.hlines(
        y=y,
        xmin=0,
        xmax=allowed,
        color=positive_color,
        linewidth=14,
        zorder=1
    )

    # Красная зона:
    # после границы ценового эффекта уже недостаточно
    ax.hlines(
        y=y,
        xmin=allowed,
        xmax=x_max,
        color=negative_color,
        linewidth=14,
        zorder=1
    )

    # Фактическое снижение платежей
    ax.scatter(
        actual,
        y,
        s=220,
        color=actual_color,
        edgecolor="white",
        linewidth=1.5,
        zorder=3
    )

    # Граница положительного эффекта
    ax.vlines(
        x=allowed,
        ymin=y - 0.11,
        ymax=y + 0.11,
        color=limit_color,
        linewidth=4,
        zorder=3
    )

    # Подпись фактического снижения
    ax.annotate(
        f"снизилось на {actual:.2f}%",
        xy=(actual, y),
        xytext=(-12, 16),
        textcoords="offset points",
        ha="right",
        va="bottom",
        fontsize=10,
        color=actual_color,
        fontweight="bold"
    )

    # Подпись границы
    ax.annotate(
        f"граница {allowed:.2f}%",
        xy=(allowed, y),
        xytext=(0, 20),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=10,
        color=limit_color,
        fontweight="bold"
    )

    # Подпись зелёной зоны
    ax.text(
        allowed / 2,
        y + 0.22,
        "повышение цены компенсирует\nснижение платежей",
        ha="center",
        va="bottom",
        fontsize=9,
        color=positive_text,
        fontweight="bold"
    )

    # Подпись красной зоны
    ax.text(
        (allowed + x_max) / 2,
        y + 0.22,
        "ценового эффекта\nуже недостаточно",
        ha="center",
        va="bottom",
        fontsize=9,
        color=negative_text,
        fontweight="bold"
    )


# Названия тарифов
ax.set_yticks(
    [1, 0],
    ["Базовый", "Расширенный"]
)

# Даём место для подписей сверху и снизу
ax.set_ylim(-0.35, 1.55)

# Общая шкала X
ax.set_xlim(0, x_max)

ax.set_xlabel(
    "Снижение числа платежей, %",
    fontsize=12
)

ax.set_ylabel(
    "Тариф",
    fontsize=12
)

# Заголовок выравниваем по левому краю области графика
fig.suptitle(
    "В исследуемых окнах снижение платежей остаётся в зоне положительного ценового эффекта",
    x=0.125,
    ha="left",
    fontsize=15,
    fontweight="bold",
    y=0.98
)

# Сетка только по X
ax.grid(
    axis="x",
    alpha=0.22,
    linewidth=0.8
)

ax.grid(
    axis="y",
    visible=False
)

sns.despine()

# Оставляем место под общий заголовок
plt.tight_layout(rect=[0, 0, 1, 0.94])

plt.show()

В обоих тарифах фактическое снижение числа платежей остаётся внутри зелёной зоны, где повышение цены компенсирует потерю платёжного объёма. Для «Базового» тарифа платежи снизились на 0,86% при рассчитанной границе около 20,08%, для «Расширенного» — на 0,73% при границе около 11,14%.

В рамках исследуемых 28-дневных окон это означает, что наблюдаемое снижение платёжного объёма недостаточно велико, чтобы перекрыть положительный эффект новой цены. То есть, по обоим тарифам повышение цен остаётся экономически успешным с точки зрения выручки.

### Выводы

В исследуемых 28-дневных окнах повышение цен было успешным по критерию выручки. После изменения цен выручка mobile-сегмента выросла с 28,13 млн руб. до 32,98 млн руб., то есть на 17,2%. При этом число успешных платежей снизилось лишь с 102 267 до 101 436, примерно на 0,81%.

Таким образом, рост выручки не был получен ценой резкого сокращения платёжного объёма: после повышения цены платежей стало немного меньше, но дополнительная выручка от более высокой цены существенно перекрыла эту потерю.

У «Базового» тарифа средний платёж вырос с 199 до 249 руб., при этом число платежей снизилось только на 0,86%. У «Расширенного» средний платёж вырос с 399 до 449 руб., а число платежей снизилось на 0,73%.

Следовательно, оба тарифа после повышения цены принесли больше выручки, несмотря на небольшое снижение количества платежей.

Факторная декомпозиция подтверждает, что основной источник роста — именно изменение цены, а не увеличение количества платежей.

Для «Базового» тарифа изменение выручки составило +3,03 млн руб. При этом снижение числа платежей дало отрицательный эффект около −0,11 млн руб., а повышение цены — положительный эффект +3,14 млн руб.

Для «Расширенного» тарифа выручка увеличилась на +1,82 млн руб.: эффект уменьшения числа платежей составил около −0,11 млн руб., тогда как ценовой эффект — +1,93 млн руб.

То есть гипотеза о том, что повышение цены перекроет возможное снижение платёжного объёма, в исследуемом периоде подтверждается для обоих тарифов. Отрицательный эффект снижения количества платежей есть, но он значительно меньше положительного ценового эффекта.

Расчёт предельно допустимого снижения платёжного объёма показывает, что наблюдаемые значения находятся далеко от границы, при которой повышение цены перестало бы компенсировать потерю платежей.

Для «Базового» тарифа при новой цене выручка сохраняется не ниже прежнего уровня, пока суммарное снижение числа платежей не превышает примерно 20,08%. Фактически в исследуемом окне оно составило только 0,86%.

Для «Расширенного» тарифа соответствующая граница ниже — 11,14%, тогда как фактическое снижение составило 0,73%.

Следовательно, гипотеза о том, что фактическое сокращение платёжного объёма остаётся в пределах, при которых повышение цены даёт положительный эффект на выручку, также подтверждается для обоих тарифов. При этом «Расширенный» тариф более чувствителен: его допустимая граница заметно ниже, поэтому именно его динамику следует контролировать внимательнее.

Выраженного канала, который уже ставил бы результат повышения цен под угрозу, не обнаружено. Наиболее заметное снижение числа платежей наблюдается в social: 2,68% для «Базового» тарифа и 3,21% для «Расширенного». Однако даже эти значения остаются существенно ниже соответствующих границ 20,08% и 11,14%.

В ряде каналов динамика ещё устойчивее: для «Расширенного» тарифа в crm и referral число платежей немного выросло, а в organic практически не изменилось.

Поэтому гипотеза о наличии канала, в котором после повышения цены платёжный объём уже снизился настолько, что угрожает положительному эффекту новой цены, не подтверждается. При этом social является наиболее чувствительным каналом и требует дальнейшего наблюдения.



По результатам исследования оснований откатывать повышение цен не видно. В выбранных равных 28-дневных окнах новая цена дала существенный прирост выручки.

Рекомендуется сохранить текущую ценовую стратегию и продолжить мониторинг трёх показателей: выручки, количества успешных платежей и фактического снижения платёжного объёма относительно рассчитанной границы. В первую очередь стоит следить за «Расширенным» тарифом, поскольку его граница ниже, и за каналом social, где наблюдается наиболее заметное сокращение платежей.

Ограничение вывода: исследование показывает результат только для зафиксированного mobile-сегмента и двух выбранных 28-дневных окон. Оно подтверждает положительный эффект повышения цены на выручку, но не доказывает долгосрочную прибыльность решения, отсутствие будущего оттока или сохранение эффекта на более длинном горизонте. 